# 05 - Inversion

§11.2 steps 9, 10 and 11:

- **step 9a**, the gradient check. Three ways of computing the same derivative, agreeing to
  `GATE_GRAD_SIGFIGS = 3` significant figures in double precision. All three differentiate the
  *surrogate*, so this is a correctness check on the autodiff graph and not on the physics.
- **step 9b**, the physical-sensitivity gate, which is the one the transfer claim rests on:
  the same receiver-field derivatives finite-differenced through the **reference solver** and
  compared in magnitude (`GATE_SENSITIVITY_REL`) and direction (`GATE_SENSITIVITY_COSINE`).
  It costs `1 + 2 P` FDTD solves per step size, so `QUICK` sweeps one `h` and the full pass three.
- **step 9c**, the same gate at each interface width `cfg.EPS_INVERT_ANNEAL` would visit. This is
  what licenses -- or refuses -- the anneal, which is off precisely because nothing had measured
  its endpoints.
- **step 10**, one noiseless inversion, end to end, with every stage's trace, and the misfit
  landscape figure §11.3 calls non-negotiable.
- **step 11**, the success-rate statistic at 30 dB over many cases, split by whether the
  illumination was held out of training, plus an SNR sweep.

There is also a cycle-skipping demonstration, which is not on the checklist but is the reason
the pipeline has four stages instead of one. It is cheap: one extra inversion started from a
deliberately bad guess with the screen disabled.

**Runtime.** Minutes for steps 9a and 10; step 9b adds `1 + 2 P len(h)` full FDTD solves on the
`376^2` padded grid (7 solves under `QUICK`, 19 on the full pass), step 9c adds 21 more at one
step size and three widths, and the verification re-solves add three; the statistics scale with
`N_CASES` and the SNR sweep. Set `QUICK = True` for a short pass. Headless:

```
modal run modal_app.py::inversion_stats
```

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

In [ ]:
# The three splits and the checkpoint the later notebooks read.  Nothing here writes.
paths = {k: E.datasets / f"{k}.h5" for k in ("train", "val", "test")}
for k, p in paths.items():
    print(f"{k:>6}  {'present' if p.exists() else 'MISSING':>7}  "
          f"{(p.stat().st_size / 1e9 if p.exists() else 0):6.2f} GB  {p}")

CKPT_DIR = E.checkpoints / "full"
CKPT = CKPT_DIR / "best.pt"
print(f"\nckpt   {'present' if CKPT.exists() else 'MISSING':>7}  {CKPT}")

In [ ]:
QUICK = True

N_CASES = 12 if QUICK else 40        # step 11, at 30 dB
N_SNR_CASES = 6 if QUICK else 16     # per SNR in the sweep
MAP_N = 31 if QUICK else 41          # misfit map resolution
SENS_H = (0.02,) if QUICK else None   # step 9b step sizes; None = SENS.SOLVER_H_SWEEP
SENS_H_EPS = (0.02,)                  # step 9c: one h, because the axis on test is eps
print(f"{'QUICK' if QUICK else 'FULL'}: {N_CASES} cases at 30 dB, "
      f"{N_SNR_CASES} per SNR over {cfg.SNR_DB_SWEEP}, {MAP_N}x{MAP_N} misfit maps")

In [ ]:
import copy

import h5py

from src import training
from src.data.dataset import load_incident, load_inversion_case
from src.geometry.sdf import Circle, net_coords, soft_indicator
from src.inverse import invert as INV
from src.inverse import sensitivity as SENS
from src.inverse.misfit import (InverseCase, Objective, SurrogateForward,
                                basin_width, misfit_map)
from src.models.fno2d import to_double

assert paths["test"].exists() and CKPT.exists(), "run notebooks 02 and 03 first"

model, meta = training.load(CKPT, device=DEV)
inc = load_incident(str(paths["test"]), device=DEV)
fwd = SurrogateForward(model, inc, device=DEV)
family = Circle()

print(f"model {meta['arch']}, epoch {meta['epoch']}")
print(f"incident cache: phasors {tuple(inc['phasors'].shape)}  "
      f"scale {tuple(inc['scale'].shape)}  scale_recv {tuple(inc['scale_recv'].shape)}")
print(f"forward frozen: {not any(p.requires_grad for p in fwd.model.parameters())}")

with h5py.File(paths["test"], "r") as f:
    src_all = f["samples/src_idx"][:]
    theta_all = f["samples/theta"][:]
    n_test = int(f.attrs["n_samples"])
held_mask = np.isin(src_all, cfg.SRC_HELDOUT)
print(f"\ntest split: {n_test} samples, {held_mask.sum()} on held-out sources "
      f"{cfg.SRC_HELDOUT}")

## Step 9a -- the geometry derivative

`d chi / d theta` for a circle, three ways: the closed form of §8.2, autodiff through
`sigmoid(-phi/eps)`, and central differences. The closed form is

    d chi / d xc = sigma' (1/eps) cos(alpha)
    d chi / d yc = sigma' (1/eps) sin(alpha)
    d chi / d R  = sigma' (1/eps)

with `sigma'` sharply peaked on the boundary. Two things are worth seeing rather than
asserting. The support is an annulus of width about `eps`, so **all** sensitivity to the
geometry lives within a couple of cells of the boundary -- which is why the interface width is
annealed rather than fixed, and why the position estimate cannot be sharper than the interface
it is estimating. And the three derivatives have distinct angular signatures: the radius
derivative is uniform round the annulus (a breathing monopole), the position derivatives carry
`cos` and `sin` (dipoles). Three parameters, three orthogonal modes -- which is the reason all
three are separately identifiable from one ring of data.

This is checked before the network is involved at all. If the geometry derivative is wrong,
every gradient in the pipeline is wrong, and no amount of inspecting the optimiser would say so.

In [ ]:
yy, xx = net_coords(device=DEV, dtype=torch.float64)
eps_len = cfg.EPS_INTERFACE_CELLS * cfg.DX_NET
i0 = int(np.where(held_mask)[0][0]) if held_mask.any() else 0
theta0 = torch.tensor(theta_all[i0], dtype=torch.float64, device=DEV).unsqueeze(0)
print(f"at theta = {np.round(_np(theta0)[0], 5)}  (test sample {i0}, "
      f"source {src_all[i0]})")


def chi_of(t):
    return soft_indicator(family.sdf(t, yy, xx), eps_len)


ana = family.dchi_dtheta_analytic(theta0, yy, xx, eps_len)[0]       # [3, ny, nx]

# autodiff, as a VJP against a fixed random probe: <v, dchi/dtheta_i>
g = torch.Generator(device="cpu").manual_seed(0)
v = torch.randn(cfg.N_NET, cfg.N_NET, generator=g,
                dtype=torch.float64).to(DEV)
t_ad = theta0.clone().requires_grad_(True)
(chi_of(t_ad) * v).sum().backward()
vjp_ad = _np(t_ad.grad[0])
vjp_ana = _np((ana * v).sum(dim=(-2, -1)))

rows = []
for i, nm in enumerate(family.param_names):
    with torch.no_grad():
        h = 1e-7
        e = torch.zeros_like(theta0)
        e[0, i] = h
        fd = (chi_of(theta0 + e) - chi_of(theta0 - e)) / (2 * h)
        fd_v = float((fd[0] * v).sum())
    a, b, c = float(vjp_ana[i]), float(vjp_ad[i]), fd_v
    dig = lambda p, q: -math.log10(max(abs(p - q) / max(abs(p), abs(q), 1e-300), 1e-300))
    rows.append((nm, f"{a:+.8e}", f"{b:+.8e}", f"{c:+.8e}",
                 f"{dig(a, b):.1f}", f"{dig(a, c):.1f}"))
table(rows, ["param", "analytic", "autodiff", "central FD",
             "digits ana/ad", "digits ana/FD"])

In [ ]:
a_np = _np(ana)
chi_np = _np(chi_of(theta0)[0])
fig, ax = plt.subplots(1, 4, figsize=(12.0, 2.9))
for i, nm in enumerate(family.param_names):
    m = np.abs(a_np[i]).max()
    im = ax[i].imshow(a_np[i], origin="lower", cmap="RdBu_r", vmin=-m, vmax=m)
    ax[i].contour(np.arange(cfg.N_NET), np.arange(cfg.N_NET), chi_np,
                  levels=[0.5], colors="k", linewidths=0.8)
    ax[i].set(title=f"d chi / d {nm}", xticks=[], yticks=[])
    ax[i].grid(False)
    fig.colorbar(im, ax=ax[i], fraction=0.046)

# radial profile of the R-derivative, against the annulus width
c_ = _np(theta0)[0]
r_pix = _np(torch.sqrt((xx - c_[0]) ** 2 + (yy - c_[1]) ** 2))
w10 = cfg.EPS_TRANSITION_FACTOR * cfg.EPS_INTERFACE_CELLS   # 10-90%, net cells
ax[3].plot(((r_pix - c_[2]) / cfg.DX_NET).ravel(), a_np[2].ravel(), ".", ms=1.5,
           alpha=0.35)
ax[3].axvspan(-0.5 * w10, 0.5 * w10, color="C3", alpha=0.12,
              label=f"10-90% width = {w10:.2f} cells")
ax[3].axvline(-cfg.EPS_INTERFACE_CELLS, ls="--", c="C3", lw=1.0)
ax[3].axvline(cfg.EPS_INTERFACE_CELLS, ls="--", c="C3", lw=1.0,
              label=f"+/- eps = {cfg.EPS_INTERFACE_CELLS:.4f} cells")
ax[3].set(xlim=(-3, 3), xlabel="distance from boundary (net cells)",
          ylabel="d chi / d R",
          title="all sensitivity is in the annulus,\nand the annulus is sub-cell")
ax[3].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_dchi_dtheta.png")
plt.show()

## Step 9b -- `dJ/dtheta` through the surrogate

Autodiff against central differences of the objective itself, in **double precision** and
**away from the minimum**.

Both of those are load-bearing. In float32 the field carries about seven digits, and a central
difference of a relative misfit built from that field loses roughly half of them before the
ratio is formed, leaving less than the three significant figures the gate asserts -- so the
check would fail for precision reasons and get quietly disabled. `to_double` exists because
`nn.Module.double()` skips the complex spectral weights. And at `theta_true` the gradient is
approximately zero, where a relative comparison of two small numbers reports noise; the check
is done at `theta_true + delta` with `delta` about a fifth of a shear wavelength, where the
gradient is O(1) and both methods have something to agree about.

The step size is swept rather than guessed. Central differences have error
`~ h^2 f''' + eps_machine f / h`, so the agreement is a V in `h`: truncation-limited on the
right, round-off-limited on the left. The reported number is the bottom of the V. A monotone
curve with no minimum would mean the objective is not smooth at this point, which would be a
finding rather than a passing check.

**What this check does not establish.** Both numbers come from differentiating *the surrogate*,
one by the chain rule and one by finite differences. Agreement to three figures says the
autodiff graph is correct -- a real and easy thing to get wrong through `soft_indicator`, the
frequency-folded batch axis, and the complex spectral weights -- and says nothing whatever about
whether the surrogate's sensitivity to a geometry perturbation matches the *medium's*. That is a
second, independent gate, and it is the next cell rather than future work. An inversion driven by
a correctly-differentiated but physically wrong sensitivity converges confidently to the wrong
geometry, and only the second gate can see it.

`eps_cells` is held at `EPS_INTERFACE_CELLS`, the width the network was trained at, for the same
reason: a gradient evaluated at an interface width never seen in training is a gradient of the
surrogate's extrapolation.

In [ ]:
case0 = InverseCase.from_dict(load_inversion_case(str(paths["test"]), i0)).to(DEV)
case64 = InverseCase(d_obs=case0.d_obs.to(torch.complex128), src_idx=case0.src_idx,
                     nu_idx=case0.nu_idx,
                     theta_true=case0.theta_true.to(torch.float64))
fwd64 = SurrogateForward(to_double(copy.deepcopy(model)), inc, device=DEV,
                         dtype=torch.float64)
obj64 = Objective(fwd64, case64, family, band=cfg.BAND_STAGE3,
                  eps_cells=cfg.EPS_INTERFACE_CELLS)

lam_s = case64.lambda_s
delta = torch.tensor([0.2 * lam_s, -0.15 * lam_s, 0.1 * lam_s],
                     dtype=torch.float64, device=DEV)
theta_g = (case64.theta_true.to(DEV) + delta).unsqueeze(0)
print(f"theta_true = {np.round(_np(case64.theta_true), 5)}")
print(f"theta_test = {np.round(_np(theta_g)[0], 5)}   "
      f"(offset {float(delta[:2].norm())/lam_s:.2f} lambda_s in position)")

t_ad = theta_g.clone().requires_grad_(True)
j0 = obj64.residual(t_ad)
j0.backward()
grad_ad = _np(t_ad.grad[0])
print(f"\nJ = {float(j0):.10e}")
print(f"autodiff dJ/dtheta = {np.array2string(grad_ad, precision=8)}")

In [ ]:
def fd_grad(h):
    out = np.zeros(3)
    with torch.no_grad():
        for i in range(3):
            e = torch.zeros_like(theta_g)
            e[0, i] = h
            out[i] = float(obj64.residual(theta_g + e)
                           - obj64.residual(theta_g - e)) / (2 * h)
    return out


hs = np.array([1e-2, 3e-3, 1e-3, 3e-4, 1e-4, 3e-5, 1e-5, 3e-6, 1e-6, 3e-7]) * lam_s
digs = []
for h in hs:
    gfd = fd_grad(float(h))
    rel = np.abs(gfd - grad_ad) / np.maximum(np.abs(grad_ad), 1e-300)
    digs.append(-np.log10(np.maximum(rel, 1e-300)))
digs = np.array(digs)                      # [n_h, 3]

best_i = int(digs.min(axis=1).argmax())
best_h = float(hs[best_i])
best = fd_grad(best_h)
worst_digits = float(digs[best_i].min())

table([(nm, f"{grad_ad[i]:+.8e}", f"{best[i]:+.8e}", f"{digs[best_i, i]:.2f}")
       for i, nm in enumerate(family.param_names)],
      ["param", "autodiff", f"central FD (h = {best_h:.2e})", "digits"])

print(f"\nworst component agrees to {worst_digits:.2f} significant figures")
print(f"gate GATE_GRAD_SIGFIGS = {cfg.GATE_GRAD_SIGFIGS}: "
      f"{'PASS' if worst_digits >= cfg.GATE_GRAD_SIGFIGS else 'FAIL'}")
assert worst_digits >= cfg.GATE_GRAD_SIGFIGS, (
    "gradient check failed -- do not run an inversion against this gradient")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.2))
for i, nm in enumerate(family.param_names):
    ax.plot(hs / lam_s, digs[:, i], "o-", ms=3.5, lw=1.0, label=nm)
ax.axhline(cfg.GATE_GRAD_SIGFIGS, ls="--", c="C3", lw=1.0,
           label=f"gate {cfg.GATE_GRAD_SIGFIGS} digits")
ax.axvline(best_h / lam_s, ls=":", c="0.4", lw=1.0, label="best h")
ax.set(xscale="log", xlabel="h / lambda_s", ylabel="digits of agreement",
       title="central-difference step sweep\n"
             "(right: truncation error;  left: round-off)")
ax.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_gradient_check.png")
plt.show()
del fwd64, obj64
if DEV.startswith("cuda"):
    torch.cuda.empty_cache()

## Step 9b -- the physical sensitivity gate

The gate the headline claim rests on. Field inputs *enable* shape transfer; what would
*establish* it is that the surrogate's derivative with respect to the geometry is the medium's
derivative, and nothing so far has compared the two. So: one column of the receiver Jacobian per
parameter, `J_k = d u_s(theta) / d theta_k` in `C^{R x 2 x F}`, formed by the **same** central
difference applied to two different models -- the surrogate and the reference FDTD solver -- and
compared in magnitude (`GATE_SENSITIVITY_REL = 0.20`) and direction
(`GATE_SENSITIVITY_COSINE = 0.95`).

Central differences on *both* sides, rather than autodiff against FD, and that is deliberate: an
identical estimator makes the comparison isolate the physics instead of confounding it with
differentiation error, and step 9a is what licenses the transitive step from this to the gradient
the optimiser actually descends. The solver cannot be differentiated in any case -- `run` is
`@torch.no_grad()`, and a 1408-step tape over `376^2` fields is not a thing to make lightly.

**Why a sweep in `h` and not one step size.** There is no exact answer to compare against here, so
the sweep is a consistency check rather than 9a's search for a round-off floor. Both ends are
biased for physical reasons: at large `h` the secant misses the curvature of a strongly nonlinear
map, and at small `h` the moved interface is resolved by a grid that did not move with it, so the
difference is dominated by how the boundary re-quantises onto the same cells. A pass at exactly one
of three step sizes is a coincidence, which is why `gate_pass_stable` demands two.

**Direction is measured real-linearly**, `Re<a,b>/(|a||b|)`, not `|<a,b>|/(|a||b|)`. The Hermitian
version scores a pure phase rotation as perfect agreement, and a rotated Jacobian column predicts a
different arrival time -- exactly the error this gate exists to catch.

The scalar `dJ/dtheta` is reported alongside, both models scored against the *same* observation
with the same functional, because that is the quantity the optimiser sees; the columns can differ
somewhat while the descent direction survives. It is measured at the **same** `theta_true + delta`
as step 9a, and for the same reason: `dJ/dtheta = 2 Re<r, J_k>` is proportional to the residual,
and on noise-free data generated by this very solver the residual at `theta_true` is a numerical
zero, so the comparison there would divide one roundoff by another. `residual_at_floor` reports
when that has happened and the scalar comparison is withheld; the Jacobian columns do not involve
the residual and would be meaningful either way.

**If this gate fails**, the transfer claim is void even though every gradient in step 9a is
correct. It is printed and recorded rather than asserted, because the failure is a finding about
the surrogate rather than a bug that invalidates the rest of the notebook -- the checklist entry
and `05_inversion.json` are what carry it.

In [ ]:
theta_9b = (case0.theta_true + delta.to(case0.theta_true)).unsqueeze(0)
sens = SENS.surrogate_vs_solver_sensitivity(
    fwd, case0, family, incident=inc, theta=theta_9b,
    **({} if SENS_H is None else dict(h_rel=SENS_H)), progress=tqdm)

print(f"{sens['n_solves']} FDTD solves at nt = {cfg.NT}, "
      f"band {sens['band']}, eps_cells = {sens['eps_cells']:.2f} "
      f"(trained width: {sens['eps_cells'] == cfg.EPS_INTERFACE_CELLS})")
print(f"evaluated at theta_true + delta, {float(delta[:2].norm())/lam_s:.2f} lambda_s "
      f"off in position -- the step 9a point")
table([(f"{d['h_rel']:.3f}", f"{d['rel_worst']:.4f}", f"{d['cosine_worst']:.4f}",
        "PASS" if (d["rel_worst"] <= sens["gate_rel"]
                   and d["cosine_worst"] >= sens["gate_cosine"]) else "FAIL")
       for d in sens["per_h"]],
      ["h / lambda_s", "rel err (worst param)", "cosine (worst param)",
       f"< {sens['gate_rel']} and > {sens['gate_cosine']}"])

In [ ]:
best_h_rel = sens["best_h_rel"]
at_best = [r for r in sens["rows"] if abs(r[1] - best_h_rel) < 1e-12]
gj = [d for d in sens["dj_dtheta"] if abs(d["h_rel"] - best_h_rel) < 1e-12][0]
table([(nm, f"{rel:.4f}", f"{cos:.4f}", f"{gj['solver'][i]:+.4e}",
        f"{gj['surrogate'][i]:+.4e}")
       for i, (nm, _, rel, cos) in enumerate(at_best)],
      ["param", "column rel err", "column cosine", "solver dJ/dtheta",
       "surrogate dJ/dtheta"])
print(f"\nat h = {best_h_rel:.3f} lambda_s (chosen by cosine over the sweep)")
if sens["residual_at_floor"]:
    print(f"gradient vector:  withheld -- the solver's residual here "
          f"({sens['misfit_solver']:.2e}) is below the\n  floor "
          f"{sens['residual_floor']:.2e}, so 2 Re<r, J> is roundoff on both sides")
else:
    print(f"gradient vector:  rel {sens['dj_rel']:.4f}   "
          f"cosine {sens['dj_cosine']:.4f}")

# Free self-test of the harness: d_obs was produced by this same solver at this same
# nt, so a solver misfit that is not O(1) at a geometry a fifth of a wavelength away
# means solver_receivers has not reproduced load_inversion_case's A-scan route.
print(f"\nmisfit at this point:  solver {sens['misfit_solver']:.3e}   "
      f"surrogate {sens['misfit_surrogate']:.3e}")

print(f"\nGATE_SENSITIVITY_REL     {sens['rel_worst']:.4f} <= "
      f"{sens['gate_rel']}: {'PASS' if sens['gate_pass_rel'] else 'FAIL'}")
print(f"GATE_SENSITIVITY_COSINE  {sens['cosine_worst']:.4f} >= "
      f"{sens['gate_cosine']}: {'PASS' if sens['gate_pass_cosine'] else 'FAIL'}")
print(f"stable over h            {sens['n_h_pass']}/{len(sens['h_rel'])} step sizes: "
      f"{'PASS' if sens['gate_pass_stable'] else 'FAIL'}")
print(f"\nstep 9b: {'PASS' if sens['gate_pass'] else 'FAIL'}")
if not sens["gate_pass"]:
    print("  -> the surrogate's geometry sensitivity is not the medium's; the shape-"
          "transfer\n     claim is not established by this checkpoint, whatever step "
          "9a says.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.2))
hh = [d["h_rel"] for d in sens["per_h"]]
ax[0].semilogx(hh, [d["cosine_worst"] for d in sens["per_h"]], "o-", ms=4, lw=1.1,
               c="C0", label="cosine (worst param)")
ax[0].axhline(sens["gate_cosine"], ls="--", c="C3", lw=1.0,
              label=f"gate {sens['gate_cosine']}")
ax[0].axvline(best_h_rel, ls=":", c="0.4", lw=1.0, label="best h")
ax[0].set(xlabel="h / lambda_s", ylabel="direction agreement",
          title="surrogate vs solver: direction", ylim=(min(0.0, *[
              d["cosine_worst"] for d in sens["per_h"]]) - 0.02, 1.02))
ax[0].legend(fontsize=7.5, loc="lower left")

x = np.arange(len(sens["param_names"]))
ax[1].bar(x - 0.2, gj["solver"], 0.4, label="reference solver", color="C0")
ax[1].bar(x + 0.2, gj["surrogate"], 0.4, label="FNO surrogate", color="C1")
ax[1].axhline(0.0, c="0.5", lw=0.8)
sub = ("residual at the floor: not compared" if sens["residual_at_floor"] else
       f"cosine {sens['dj_cosine']:.3f}, rel {sens['dj_rel']:.3f}")
ax[1].set(xticks=x, xlabel="parameter", ylabel="dJ/dtheta",
          title=f"gradient at h = {best_h_rel:.3f} lambda_s\n{sub}")
ax[1].set_xticklabels(sens["param_names"])
ax[1].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_sensitivity_gate.png")
plt.show()

## Step 9c -- the interface-width axis

`cfg.EPS_INVERT_ANNEAL` is `False`, and step 9b does not explain why: 9b runs at the trained
width and says nothing about any other. The anneal would sweep `eps` from `EPS_INVERT_START = 2.0`
cells down to `EPS_INVERT_END = 1.0` while differentiating, which asks the surrogate two questions
it was never trained on -- the endpoints -- and calls the answers gradients. Bracketing the
training value is not validation of the endpoints; it is where the anneal spends its first and its
last iterations.

So this is 9b asked again at each width the schedule would visit, and it is the measurement that
would license switching the anneal on. `SENS.eps_transfer_report` re-runs the whole comparison per
width against the same two gates, and `anneal_licensed` is the conjunction over widths. One step
size is used throughout rather than the sweep: 9b already established stability over `h` at the
trained width, and the axis under test here is `eps`. `3 * (1 + 2 P)` = 21 FDTD solves.

`degradation` is the quantity the anneal is betting against -- the worst width's magnitude error
divided by the trained width's. A value near 1 says the width does not matter; a large one says
each annealing step moves the surrogate somewhere it was never checked, and the licence is refused
whatever the endpoint gates happen to read. A refusal here is not a failure of the method: the
anneal is an optimisation convenience the pipeline currently does without, and the honest outcome
is a number in the record rather than a flag flipped on a hunch.

In [ ]:
eps_sweep = (cfg.EPS_INVERT_START, cfg.EPS_INTERFACE_CELLS, cfg.EPS_INVERT_END)
eps_rep = SENS.eps_transfer_report(
    fwd, case0, family, incident=inc, eps_cells=eps_sweep, theta=theta_9b,
    h_rel=SENS_H_EPS, progress=tqdm)

table([(f"{d['eps_cells']:.2f}", f"{d['eps_phys']:.4f}",
        "trained" if d["trained_width"] else "-",
        f"{d['rel_worst']:.4f}", f"{d['cosine_worst']:.4f}",
        "PASS" if d["gate_pass"] else "FAIL")
       for d in eps_rep["per_eps"]],
      ["eps (cells)", "eps (phys)", "width", "rel (worst param)",
       "cosine (worst param)", f"< {eps_rep['gate_rel']} and > "
       f"{eps_rep['gate_cosine']}"])

deg = eps_rep["degradation"]
print(f"\ndegradation away from the trained width: "
      f"{'-' if deg is None else f'{deg:.2f}x'}   "
      f"(1.0 would mean eps does not matter)")
print(f"anneal_licensed {eps_rep['anneal_licensed']}, and it is currently "
      f"{'ON' if eps_rep['anneal_enabled'] else 'OFF'} "
      f"(cfg.EPS_INVERT_ANNEAL)")
if eps_rep["anneal_licensed"] and not eps_rep["anneal_enabled"]:
    print("  -> every width the schedule would visit passes 9b's gates: switching\n"
          "     EPS_INVERT_ANNEAL on is licensed by this measurement.")
elif not eps_rep["anneal_licensed"]:
    print("  -> at least one width the schedule would visit fails 9b's gates, so the\n"
          "     anneal stays off.  This is the reason, not a preference.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.2))
ec = [d["eps_cells"] for d in eps_rep["per_eps"]]
o = np.argsort(ec)
ec = np.asarray(ec)[o]
rw = np.asarray([d["rel_worst"] for d in eps_rep["per_eps"]])[o]
cw = np.asarray([d["cosine_worst"] for d in eps_rep["per_eps"]])[o]

ax[0].semilogy(ec, rw, "o-", ms=5, lw=1.1, c="C0")
ax[0].axhline(eps_rep["gate_rel"], ls="--", c="C3", lw=1.0,
              label=f"gate {eps_rep['gate_rel']}")
ax[0].axvline(cfg.EPS_INTERFACE_CELLS, ls=":", c="0.4", lw=1.0, label="trained width")
ax[0].set(xlabel="eps (network cells)", ylabel="rel err, worst parameter",
          title="magnitude agreement vs interface width")
ax[0].legend(fontsize=7.5)

ax[1].plot(ec, cw, "s-", ms=5, lw=1.1, c="C1")
ax[1].axhline(eps_rep["gate_cosine"], ls="--", c="C3", lw=1.0,
              label=f"gate {eps_rep['gate_cosine']}")
ax[1].axvline(cfg.EPS_INTERFACE_CELLS, ls=":", c="0.4", lw=1.0, label="trained width")
ax[1].set(xlabel="eps (network cells)", ylabel="cosine, worst parameter",
          title="direction agreement vs interface width",
          ylim=(min(0.0, cw.min()) - 0.02, 1.02))
ax[1].legend(fontsize=7.5, loc="lower left")
fig.tight_layout()
savefig(fig, "05_eps_transfer.png")
plt.show()

## Step 10 -- one inversion, noiseless

Four stages, and each exists because the one before it cannot do its job:

| stage | what it does | band | why |
|-------|--------------|------|-----|
| 0 | RingCNN guess (notebook 06) | - | free, and right when the defect is in-distribution |
| 1 | 256-candidate envelope screen, 16 kept | `1..6` | narrow band, so the widest basin |
| 2 | Adam, 200 steps, all survivors at once | `1..10` | far from the optimum, where a curvature model is a liability |
| 3 | L-BFGS strong-Wolfe, complex misfit | full | locally quadratic, and the only stage that uses carrier phase |

No stage hard-codes its objective: stage 1 reads `cfg.SCREEN_OBJECTIVE`, stages 2 and 3 read
`cfg.STAGE_OBJECTIVE`, and `res.stages["objectives"]` records what actually ran. The v2.0
pipeline screened on the **magnitude spectrum**, which was a mistake of a specific and instructive
kind: `|g_hat(w) e^{-i w tau}| = |g_hat(w)|` exactly, so that objective does not merely blur
travel-time information, it is *identically blind* to it. Its "wide basin" was a flat plateau. It
survives as `cfg.SCREEN_FALLBACK_OBJECTIVE` for reproducing the old numbers, and
`INV.screen_capture_rate` measures both against `GATE_SCREEN_CAPTURE`.

Stage 1 keeps 16 survivors rather than 1 because the screen steps `0.91 lambda_s` across the
interior -- about eight times the `lambda_s/4` waveform basin -- so the true optimum falls
between nodes and the nearest few are all mediocre and nearly tied. Picking one would be a coin
flip.

Stage 3 holds `eps` fixed at the trained width. `cfg.EPS_INVERT_ANNEAL` is `False`, and step 9c
above is the reason it is a measurement rather than a preference: a schedule from 2.0 down to 1.0
cells asks the network questions it was never trained on at both endpoints, and `anneal_licensed`
reports whether the reference solver agrees with the surrogate's sensitivities at each of them.
When it is switched on, `INV.eps_schedule_default()` returns the three widths and L-BFGS rebuilds
its history at each one -- the objective changes when `eps` does, and curvature estimated on the
old objective is worse than no curvature at all.

In [ ]:
res = INV.invert(fwd, case0, family=family, log=True)
print()
print(res.summary())
st = res.stages
print(f"\nstage 1 best J {st['stage1_best_J']:.4e}   "
      f"stage 2 J {st['stage2_J']:.4e}   stage 3 J {st['stage3_J']:.4e}")
print(f"stage 2 started from {len(st['stage1_J'])} survivors, "
      f"kept theta {np.round(st['stage2_theta'], 4)}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11.4, 3.0))

j1 = np.asarray(st["stage1_J"])
ax[0].plot(np.arange(len(j1)), j1, "o-", ms=4)
ax[0].set(xlabel="survivor rank", ylabel=f"{st['objectives'][1]} misfit",
          title=f"stage 1: {st['stage1_n_candidates']} candidates,\n"
                f"{cfg.N_SURVIVORS} kept -- note how nearly tied")

ax[1].semilogy(st["stage2_trace"], lw=1.1)
ax[1].set(xlabel="Adam step", ylabel="mean J over survivors",
          title=f"stage 2: {st['objectives'][2]}, band 1..{cfg.BAND_STAGE2.stop}, "
                f"lr {cfg.ADAM_LR_STAGE2}")

tr3 = np.asarray(st["stage3_trace"])
ax[2].semilogy(tr3, lw=1.1)
eps_sched = st["eps_schedule"]
n_eps = len(eps_sched)
for b in range(1, n_eps):
    ax[2].axvline(b * len(tr3) / n_eps, ls=":", c="0.45", lw=1.0)
eps_txt = (f"eps fixed at {eps_sched[0]} cells (trained width)" if n_eps == 1
           else f"eps {eps_sched[0]} -> {eps_sched[-1]} cells"
                "\n(dotted: fresh history per eps)")
ax[2].set(xlabel="closure evaluation", ylabel="J + Tikhonov",
          title=f"stage 3: L-BFGS, {st['objectives'][3]}\n{eps_txt}")
fig.tight_layout()
savefig(fig, "05_stage_traces.png")
plt.show()

## Figure 4 -- the misfit landscape

`J` over `(xc, yc)` at fixed `R`, computed twice on the same case:

- **left**, the stage-1 objective (`cfg.SCREEN_OBJECTIVE`) over the lowest 6 lines;
- **right**, the stage-3 objective (complex misfit) over the full band.

Two features are being looked for, and both are physics rather than decoration.

The basin should be about `lambda_s/4` across -- the resolution a half-wavelength criterion
predicts -- and it should be elongated **transverse to the source-to-defect line**. The direction
follows from path length: displace the trial void by `d` *along* the ray and the two-way travel
path changes by about `2d`, so the phase moves immediately and the misfit climbs, which makes the
along-ray direction the **narrow** one. Displace it by `d` across the ray at range `L` and the
path changes by only `~d^2/L`, second order, so that direction is **flat and long**. v2.0 stated
this the other way round and used the same physics to argue for it, which is worth recording:
the sentence "one source constrains travel time, hence range, far better than angle" is correct
and its conclusion was inverted.

One caveat keeps the claim falsifiable. A 32-receiver ring is not one source: receivers spread
around the defect also constrain the transverse direction, so the transverse axis is longer but
not unboundedly so, and the honest prediction is the **ordering** -- `across >= along` -- rather
than a particular ratio. `basin_width` reports `elongation` so the ordering can be read off
directly.

And the right panel should show *concentric ripples* that the left panel does not. Those are
cycle skips: move the trial void half a wavelength and the modelled arrival slips by a full
period, so the complex misfit returns almost to its minimum at a place that is wrong. The
envelope screen has ripples too -- it is built from the same phasors -- but they are suppressed,
because an envelope of bandwidth `B` decorrelates over `~1/B` rather than over a carrier period.
With the lowest six lines that is `5.6 lambda_s` against `lambda_s/4`, and the widening of the
basin is what makes the screen work.

For contrast, `spectral_magnitude` (the v2.0 screen, still available) has no ripples at all -- and
no basin either. It is not a smoothed version of the right panel; it is a different function of
the data that has thrown the travel time away. The next cell measures both.

In [ ]:
obj_screen = Objective.for_stage(1, fwd, case0, family, scale_invariant=True)
obj_full = Objective.for_stage(3, fwd, case0, family)
print(f"stage 1 objective = {obj_screen.objective!r}   "
      f"stage 3 objective = {obj_full.objective!r}")

t0 = time.perf_counter()
m_screen = misfit_map(obj_screen, n=MAP_N)
m_full = misfit_map(obj_full, n=MAP_N)
print(f"two {MAP_N}x{MAP_N} maps in {time.perf_counter()-t0:.1f} s")

b_screen = basin_width(m_screen)
b_full = basin_width(m_full)
table([(f"stage 1 ({obj_screen.objective}, band 1..{cfg.BAND_STAGE1.stop})",
        f"{b_screen['along_ls']:.3f}", f"{b_screen['across_ls']:.3f}",
        f"{b_screen['elongation']:.2f}", f"{b_screen['angle_deg']:.1f}"),
       (f"stage 3 ({obj_full.objective}, full band)",
        f"{b_full['along_ls']:.3f}", f"{b_full['across_ls']:.3f}",
        f"{b_full['elongation']:.2f}", f"{b_full['angle_deg']:.1f}")],
      ["objective", "along (l_s)", "across (l_s)", "across/along",
       "source angle (deg)"])
print(f"\nlambda_s/4 = {0.25:.2f} lambda_s is the half-wavelength resolution estimate;"
      f"\nelongation > 1 is the falsifiable prediction (across the ray is the long axis)")

### Does the screen actually cover its own grid?

The basin widths above are a property of one case at one radius, so they are a measurement and
not a guarantee -- which is the whole point of replacing v2.0's "coverage is guaranteed by a
factor of nearly three". `envelope_basin_width` compares the objectives on a shared radius and
puts the screen's worst-case node distance next to the narrow objective's half-width; the
`covered` flag is that comparison, not an assumption.

In [ ]:
from src.inverse.misfit import envelope_basin_width

cov = envelope_basin_width(obj_full, n=MAP_N)
table([(k, f"{cov['per_objective'][k]['along_ls']:.3f}",
        f"{cov['per_objective'][k]['across_ls']:.3f}",
        f"{cov['per_objective'][k]['elongation']:.2f}")
       for k in cov["per_objective"]],
      ["objective", "along (l_s)", "across (l_s)", "across/along"])
print(f"\nscreen step            {cov['screen_step_ls']:.3f} lambda_s")
print(f"worst node distance    {cov['worst_node_distance_ls']:.3f} lambda_s")
print(f"narrow half-width      {cov['narrow_half_width_ls']:.3f} lambda_s")
print(f"margin                 {cov['margin']:.2f}x      covered = {cov['covered']}")
print(f"\nv2.0 nominal N_c/2 factor {cov['nominal_v2_ratio']:.1f}x, "
      f"measured {cov['measured_ratio']:.2f}x")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.6, 4.2))
for a_, m, ttl in [(ax[0], m_screen,
                    f"stage 1: {obj_screen.objective}, band 1..{cfg.BAND_STAGE1.stop}"),
                   (ax[1], m_full, f"stage 3: {obj_full.objective}, full band")]:
    J = _np(m["J"])
    x, y = _np(m["x"]), _np(m["y"])
    im = a_.pcolormesh(x, y, np.log10(J), shading="nearest", cmap="viridis")
    a_.contour(x, y, J, levels=[2.0 * J.min()], colors="w", linewidths=1.2)
    tt = _np(m["theta_true"])
    sx_, sy_ = m["source_xy"]
    a_.plot(tt[0], tt[1], "r*", ms=13, label="truth")
    a_.plot(*m["argmin"], "wx", ms=8, mew=2, label="grid argmin")
    a_.plot(_np(res.theta)[0], _np(res.theta)[1], "co", ms=6, mfc="none", mew=1.6,
            label="inverted")
    a_.plot([sx_, tt[0]], [sy_, tt[1]], "-", c="w", lw=0.8, alpha=0.7)
    a_.plot(sx_, sy_, "w^", ms=8, label="source")
    ls = m["lambda_s"]
    a_.plot([x[1], x[1] + 0.25 * ls], [y[1], y[1]], "-", c="w", lw=3)
    a_.text(x[1], y[1] + 0.06 * ls, "lambda_s/4", color="w", fontsize=7.5)
    a_.set(xlabel="xc", ylabel="yc", title=f"{ttl}\nlog10 J at R = {m['radius']:.4f}",
           aspect="equal")
    a_.grid(False)
    a_.legend(fontsize=7, loc="upper right", framealpha=0.7)
    fig.colorbar(im, ax=a_, fraction=0.046)
fig.tight_layout()
savefig(fig, "05_fig4_misfit_landscape.png")
plt.show()

### The same thing as a 1-D slice

Along the source-to-defect line, which is the direction cycle skips live in -- and, per the
argument above, the **narrow** direction of the basin. The complex misfit should oscillate with a
period near `lambda_s/2`, since half a wavelength of position moves the two-way path by a full
wavelength. The stage-1 envelope should show the same oscillation *strongly damped*: it is built
from the same phasors, so the carrier is still in there, but a `0.179 f_c` band smooths over it.
Every local minimum in the blue curve is a place a gradient-based inversion started on the wrong
side of would happily converge to and report a confident answer.

What to look for is therefore a difference of degree, and the number to read off is how many
local minima each curve has. If the orange curve is *perfectly* flat, that is not success -- it
means the objective in use has no travel-time content at all, which is the `spectral_magnitude`
failure, and the screen would be ranking candidates on amplitude pattern alone.

In [ ]:
tt = _np(case0.theta_true)
sx_, sy_ = cfg.SOURCE_XY[case0.src_idx]
ang = math.atan2(tt[1] - sy_, tt[0] - sx_)
s_ = np.linspace(-1.6, 1.6, 161) * case0.lambda_s
th_line = torch.tensor(
    np.stack([tt[0] + s_ * math.cos(ang), tt[1] + s_ * math.sin(ang),
              np.full_like(s_, tt[2])], axis=-1),
    dtype=torch.float32, device=DEV)

with torch.no_grad():
    js, jf = [], []
    for k in range(0, th_line.shape[0], cfg.SCREEN_CHUNK):
        c = th_line[k:k + cfg.SCREEN_CHUNK]
        js.append(_np(obj_screen.residual(c)))
        jf.append(_np(obj_full.residual(c)))
js, jf = np.concatenate(js), np.concatenate(jf)

fig, ax = plt.subplots(figsize=(7.0, 3.2))
ax.semilogy(s_ / case0.lambda_s, jf, lw=1.2, c="C0",
            label="complex, full band (stage 3)")
ax.semilogy(s_ / case0.lambda_s, js, lw=1.2, c="C1",
            label=f"{obj_screen.objective}, band 1..{cfg.BAND_STAGE1.stop} (stage 1)")
loc = [i for i in range(1, len(jf) - 1) if jf[i] < jf[i-1] and jf[i] < jf[i+1]]
loc_s = [i for i in range(1, len(js) - 1) if js[i] < js[i-1] and js[i] < js[i+1]]
ax.plot(s_[loc] / case0.lambda_s, jf[loc], "v", ms=5, c="C3",
        label=f"{len(loc)} local minima (complex) vs {len(loc_s)} (stage 1)")
ax.axvline(0.0, ls="--", c="0.4", lw=1.0, label="truth")
for k in (-1.0, -0.5, 0.5, 1.0):
    ax.axvline(k * 0.5, ls=":", c="0.75", lw=0.8)
ax.set(xlabel="displacement along the source-defect line (lambda_s)",
       ylabel="J", title="cycle skipping: dotted lines every lambda_s/2")
ax.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_cycle_skipping.png")
plt.show()

### What happens if you skip the screen

`skip_screen=True` with a starting guess a wavelength away, which is what a single-stage
gradient inversion amounts to. The screen exists because this is the alternative -- and note
that the failure is not obvious from the outside: the final misfit is small, the optimiser
reports convergence, and the answer is wrong by more than a wavelength.

That last observation cuts both ways, and it is worth being blunt about which way. It shows the
final misfit is **not** a certificate: a converged cycle-skipped inversion can leave a residual
comparable to a correct one, so a small residual does not establish that the geometry is right.
The `lack_of_fit` statistic of notebook 06 is therefore a one-sided instrument -- a large value is
evidence that *something* is unmodelled, without saying what -- and the thing that actually
catches this failure is re-solving at the recovered geometry with the reference solver
(`inverse/sensitivity.py:verify_with_solver`, gated by `GATE_SOLVER_VERIFY_LS`, and run on both
answers immediately below), because a surrogate error the optimiser has learned to exploit does
not survive an independent forward solve.

In [ ]:
lo_b, hi_b = family.bounds(case0.lambda_s)
lo_b, hi_b = lo_b.to(DEV), hi_b.to(DEV)
off = 0.75 * case0.lambda_s          # between two cycle-skip minima, not at one
bad = case0.theta_true.clone()
bad[0] += off * math.cos(ang)
bad[1] += off * math.sin(ang)
bad = torch.clamp(bad, lo_b + 1e-3, hi_b - 1e-3)     # bounds are enforced anyway
d_start = float((bad[:2] - case0.theta_true[:2]).norm()) / case0.lambda_s

res_bad = INV.invert(fwd, case0, family=family, theta_init=bad.to(DEV),
                     skip_screen=True)
print(f"start        {np.round(_np(bad), 4)}  ({d_start:.2f} lambda_s from truth, "
      f"along the source-defect line)")
print(f"converged to {np.round(_np(res_bad.theta), 4)}")
print(f"truth        {np.round(_np(case0.theta_true), 4)}")
table([("with the screen", f"{res.position_error_ls:.4f}", f"{res.misfit:.4e}",
        "PASS" if res.success else "FAIL"),
       ("screen skipped, bad start", f"{res_bad.position_error_ls:.4f}",
        f"{res_bad.misfit:.4e}", "PASS" if res_bad.success else "FAIL")],
      ["run", "position error (l_s)", "final misfit", f"gate < {cfg.GATE_POSITION_LS}"])
print(f"\nthe bad run moved {float((res_bad.theta[:2]-bad[:2]).norm())/case0.lambda_s:.3f}"
      f" lambda_s from where it started, and stopped "
      f"{res_bad.position_error_ls:.3f} lambda_s from the answer")

### Step 10b -- what the reference solver makes of both answers

The claim just made -- that the independent forward solve is what catches a cycle-skipped
inversion when the misfit cannot -- is testable on the two results in hand, and costs two solver
runs each. `verify_with_solver` re-solves at the recovered geometry and reports three things the
surrogate's own residual cannot:

- `residual_ratio`, the solver's residual at the answer over its residual at the truth. Reported
  as a magnitude and not as a "the solver prefers the truth" flag, because with noise-free data
  from this same solver the truth fits better than *any* non-exact estimate: the boolean would
  read True for every inversion ever run. It is withheld entirely when the truth's own residual
  sits at the numerical floor, which is exactly what happens here -- `d_obs` came from this solver
  at this geometry, so the denominator is a roundoff and the ratio is unbounded for a right answer
  and a wrong one alike. What remains is the comparison the ratio was for: the two answers against
  each other, and both against the surrogate.
- `surrogate_optimism`, the solver's residual at the answer over the surrogate's. Large means the
  optimiser found a hole in the model rather than a defect in the specimen.
- `misfit_solver_truth`, which on noise-free data is that floor, and is worth printing precisely
  because it shows how little the reported misfit has to beat.

What is gated is the position error, so this check exists only where ground truth does. On real
data `gate_pass` is `None` -- not `True` -- because a `None` that reads as a pass is how a
validation suite comes to certify data it never checked.

In [ ]:
ver = SENS.verify_with_solver(res, case0, incident=inc, family=family, forward=fwd)
ver_bad = SENS.verify_with_solver(res_bad, case0, incident=inc, family=family,
                                  forward=fwd)


def _ratio(v):
    return "at floor" if v["residual_ratio"] is None else f"{v['residual_ratio']:.3e}"


table([(nm, f"{v['misfit_reported']:.3e}", f"{v['misfit_solver']:.3e}", _ratio(v),
        f"{v['surrogate_optimism']:.3e}", f"{v['position_error_ls']:.4f}",
        "PASS" if v["gate_pass"] else "FAIL")
       for nm, v in [("with the screen", ver), ("screen skipped", ver_bad)]],
      ["run", "misfit (surrogate)", "misfit (solver)", "J/J_truth",
       "optimism", "position (l_s)", f"gate < {ver['gate']}"])
print(f"\nsolver residual at the truth: {ver['misfit_solver_truth']:.3e}   "
      f"(floor {ver['residual_floor']:.1e}, shared by both rows)")
print(f"the two answers differ by {float((res.theta[:2]-res_bad.theta[:2]).norm())/case0.lambda_s:.2f}"
      f" lambda_s and by {ver_bad['misfit_reported']/ver['misfit_reported']:.1f}x in "
      f"reported misfit,")
print(f"but by {ver_bad['misfit_solver']/ver['misfit_solver']:.1f}x once the reference "
      f"solver is asked -- which is the\nwhole argument for step 10b.")

## Step 11 -- the success-rate statistic

`N_CASES` test samples at 30 dB, half on trained illuminations and half on the held-out sources
`SRC_HELDOUT = (3, 6)`, so the generalisation split is balanced rather than incidental.

Noise is added to the **total** velocity A-scans in the time domain, and the incident field is
subtracted afterwards. That order matters and `load_inversion_case` enforces it: a real
instrument measures the total field, so the noise floor is set by the total amplitude, which
near the source is far larger than the scattered signal. Noising the residual instead would
make the inversion look good at SNRs where it would in fact fail.

`summarise` reports the median position error next to the mean because the failure is bimodal
rather than heavy-tailed: an inversion either lands in the right basin (`~lambda_s/20`) or
skips into a neighbouring one (`~lambda_s/2`), and a mean over that describes neither mode.

In [ ]:
rng = np.random.default_rng(cfg.SEED)
idx_held = rng.choice(np.where(held_mask)[0], size=min(N_CASES // 2,
                                                       int(held_mask.sum())),
                      replace=False)
idx_tr = rng.choice(np.where(~held_mask)[0], size=N_CASES - len(idx_held),
                    replace=False)
sel = np.concatenate([idx_tr, idx_held])
is_held = np.concatenate([np.zeros(len(idx_tr), bool), np.ones(len(idx_held), bool)])
print(f"{len(sel)} cases: {len(idx_tr)} trained sources, {len(idx_held)} held out")


def make_cases(indices, snr_db, seed=cfg.SEED):
    out = []
    for k, i in enumerate(indices):
        g = torch.Generator().manual_seed(int(seed) + int(i))
        d = load_inversion_case(str(paths["test"]), int(i), snr_db=snr_db,
                                generator=g)
        out.append(InverseCase.from_dict(d).to(DEV))
    return out


cases30 = make_cases(sel, 30.0)
t0 = time.perf_counter()
res30 = INV.run_many(fwd, cases30, family=family, progress=tqdm)
print(f"\n{len(res30)} inversions in {(time.perf_counter()-t0)/60:.1f} min "
      f"({(time.perf_counter()-t0)/len(res30):.1f} s each)")

s_all = INV.summarise(res30)
s_tr = INV.summarise([r for r, h in zip(res30, is_held) if not h])
s_he = INV.summarise([r for r, h in zip(res30, is_held) if h])
table([(k, f"{s_all[k]}", f"{s_tr[k]}", f"{s_he[k]}") for k in
       ("n", "success_rate", "position_ls_median", "position_ls_mean",
        "position_ls_p90", "radius_ls_median", "misfit_median", "seconds_mean")],
      ["", "all", "trained sources", f"held out {cfg.SRC_HELDOUT}"])
print(f"\ngate: success rate >= {cfg.GATE_SUCCESS_RATE:.0%} at 30 dB  ->  "
      f"{'PASS' if s_all['gate_pass'] else 'FAIL'} "
      f"({s_all['success_rate']:.1%})")

In [ ]:
pos30 = np.array([r.position_error_ls for r in res30])
mis30 = np.array([r.misfit for r in res30])
rad_true = np.array([float(r.theta_true[2]) for r in res30])
lam30 = np.array([r.lambda_s for r in res30])

fig, ax = plt.subplots(1, 3, figsize=(11.4, 3.0))
bins = np.linspace(0, max(pos30.max() * 1.05, cfg.GATE_POSITION_LS * 2), 30)
ax[0].hist(pos30[~is_held], bins=bins, alpha=0.75, label="trained sources")
ax[0].hist(pos30[is_held], bins=bins, alpha=0.75, label=f"held out {cfg.SRC_HELDOUT}")
ax[0].axvline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0,
              label=f"gate {cfg.GATE_POSITION_LS} lambda_s")
ax[0].set(xlabel="position error / lambda_s", ylabel="count",
          title="bimodal: right basin, or a skip")
ax[0].legend(fontsize=7)

ax[1].loglog(mis30, np.maximum(pos30, 1e-4), "o", ms=4, alpha=0.7)
ax[1].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[1].set(xlabel="final data misfit", ylabel="position error / lambda_s",
          title="correlated, not diagnostic\n(see the caveat below)")

ax[2].plot(rad_true / lam30, pos30, "o", ms=4, alpha=0.7)
ax[2].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[2].axvline(cfg.R_MIN_LS, ls=":", c="0.4", lw=1.0, label="R_MIN_LS")
ax[2].set(xlabel="true R / lambda_s", ylabel="position error / lambda_s",
          yscale="log", title="small voids are the hard ones")
ax[2].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "05_success_statistics.png")
plt.show()

r_ = np.corrcoef(np.log(np.maximum(mis30, 1e-30)),
                 np.log(np.maximum(pos30, 1e-6)))[0, 1]
print(f"corr(log misfit, log position error) = {r_:+.3f}")
print("\nRead this as a correlation and nothing stronger.  The middle panel's scatter is")
print("wide in the direction that matters: cases with a small residual and a large position")
print("error are cycle skips the residual did not notice, and they are the ones a")
print("certificate would have to catch.  Notebook 06 turns the residual into a *lack-of-fit*")
print("indicator -- one-sided, calibrated on in-family cases, threshold frozen before")
print("testing -- rather than a shape-specific mismatch detector.")

## The SNR sweep

`SNR_DB_SWEEP = (60, 40, 30, 20)`. 60 dB is effectively noiseless and measures the surrogate's
own error floor; 20 dB is where a scattered signal from a small void near the noise floor stops
being recoverable. The interesting number is not the success rate at any one SNR but where the
curve breaks, because that is what says whether the 30 dB result is comfortably inside the
working range or sitting on a cliff edge.

In [ ]:
sel_s = sel[:N_SNR_CASES]
held_s = is_held[:N_SNR_CASES]
sweep = {}
for snr in cfg.SNR_DB_SWEEP:
    cs = make_cases(sel_s, float(snr))
    rs = INV.run_many(fwd, cs, family=family)
    sweep[float(snr)] = dict(summary=INV.summarise(rs),
                             pos=[r.position_error_ls for r in rs],
                             misfit=[r.misfit for r in rs])
    s = sweep[float(snr)]["summary"]
    print(f"{snr:5.0f} dB   success {s['success_rate']:6.1%}   "
          f"median pos err {s['position_ls_median']:.4f} lambda_s   "
          f"median misfit {s['misfit_median']:.3e}")

In [ ]:
snrs = sorted(sweep, reverse=True)
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
ax[0].plot(snrs, [sweep[s]["summary"]["success_rate"] for s in snrs], "o-", ms=5)
ax[0].axhline(cfg.GATE_SUCCESS_RATE, ls="--", c="C3", lw=1.0,
              label=f"gate {cfg.GATE_SUCCESS_RATE:.0%}")
ax[0].axvline(30.0, ls=":", c="0.4", lw=1.0, label="the quoted 30 dB")
ax[0].set(xlabel="SNR (dB)", ylabel="success rate", ylim=(-0.05, 1.05),
          title=f"success vs SNR ({N_SNR_CASES} cases each)")
ax[0].invert_xaxis()
ax[0].legend(fontsize=7.5)

for s in snrs:
    p = np.maximum(np.asarray(sweep[s]["pos"]), 1e-4)
    ax[1].semilogy([s] * len(p), p, "o", ms=4, alpha=0.55)
ax[1].semilogy(snrs, [sweep[s]["summary"]["position_ls_median"] for s in snrs],
               "k-", lw=1.2, label="median")
ax[1].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[1].set(xlabel="SNR (dB)", ylabel="position error / lambda_s",
          title="per-case errors")
ax[1].invert_xaxis()
ax[1].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_snr_sweep.png")
plt.show()

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "quick": QUICK,
    "checkpoint": str(CKPT), "arch": meta["arch"],
    "gradient_check": dict(
        digits=worst_digits, gate=cfg.GATE_GRAD_SIGFIGS,
        passed=bool(worst_digits >= cfg.GATE_GRAD_SIGFIGS),
        best_h_over_lambda_s=best_h / lam_s,
        autodiff=grad_ad.tolist(), fd=best.tolist(),
        offset_lambda_s=float(delta[:2].norm()) / lam_s),
    "physical_sensitivity": {k: v for k, v in sens.items()
                             if k not in ("rows", "theta")},
    # Per width, minus each width's own full report: the nested `report` repeats
    # `physical_sensitivity` three times over and the JSON is read, not just archived.
    "eps_transfer": dict(
        eps_cells=list(eps_rep["eps_cells"]),
        anneal_licensed=eps_rep["anneal_licensed"],
        anneal_enabled=eps_rep["anneal_enabled"],
        degradation=eps_rep["degradation"],
        gate_rel=eps_rep["gate_rel"], gate_cosine=eps_rep["gate_cosine"],
        h_rel=list(SENS_H_EPS),
        per_eps=[{k: v for k, v in d.items() if k != "report"}
                 for d in eps_rep["per_eps"]]),
    "solver_verify": dict(
        screened={k: v for k, v in ver.items() if k not in ("theta", "theta_true")},
        skipped={k: v for k, v in ver_bad.items()
                 if k not in ("theta", "theta_true")}),
    "single_case": dict(
        index=int(i0), src_idx=int(case0.src_idx),
        held_out=bool(src_all[i0] in cfg.SRC_HELDOUT),
        theta_true=_np(case0.theta_true).tolist(),
        theta=_np(res.theta).tolist(), misfit=res.misfit,
        position_ls=res.position_error_ls, radius_ls=res.radius_error_ls,
        seconds=res.seconds, n_forward=res.n_forward,
        stage1_best_J=st["stage1_best_J"], stage2_J=st["stage2_J"],
        stage3_J=st["stage3_J"]),
    "skip_screen_control": dict(
        theta_start=_np(bad).tolist(), start_offset_ls=d_start,
        theta=_np(res_bad.theta).tolist(), misfit=res_bad.misfit,
        position_ls=res_bad.position_error_ls, success=res_bad.success),
    "basin": dict(screen=b_screen, full=b_full, coverage=cov,
                  objectives={1: obj_screen.objective, 3: obj_full.objective},
                  n_local_minima_along_line=len(loc),
                  n_local_minima_along_line_screen=len(loc_s)),
    "step11_30db": dict(all=s_all, trained=s_tr, heldout=s_he,
                        indices=sel.tolist(), held=is_held.tolist(),
                        position_ls=pos30.tolist(), misfit=mis30.tolist(),
                        corr_logmisfit_logerror=float(r_)),
    "snr_sweep": {str(k): v for k, v in sweep.items()},
}
dump(record, "05_inversion.json")

print()
gates = {
    f"gradient agrees to {cfg.GATE_GRAD_SIGFIGS} sig figs (9a, surrogate only)":
        worst_digits >= cfg.GATE_GRAD_SIGFIGS,
    f"sensitivity matches the solver to {cfg.GATE_SENSITIVITY_REL:.0%} (9b)":
        sens["gate_pass_rel"],
    f"sensitivity direction cosine >= {cfg.GATE_SENSITIVITY_COSINE} (9b)":
        sens["gate_pass_cosine"],
    f"and survives {'2 of 3' if len(sens['h_rel']) >= 3 else 'its one'} step sizes":
        sens["gate_pass_stable"],
    "eps anneal is off, or licensed at every width it would visit (9c)":
        bool(eps_rep["anneal_licensed"] or not eps_rep["anneal_enabled"]),
    f"single case within {cfg.GATE_POSITION_LS} lambda_s": res.success,
    f"solver agrees the answer is right (10b, < {cfg.GATE_SOLVER_VERIFY_LS} l_s)":
        bool(ver["gate_pass"]),
    "the solver rejects the cycle-skipped answer": not ver_bad["gate_pass"],
    f"success rate >= {cfg.GATE_SUCCESS_RATE:.0%} at 30 dB": s_all["gate_pass"],
    "held-out illuminations also pass": s_he.get("gate_pass", False),
    "screen covers its own grid (measured, not assumed)": cov["covered"],
    "basin is longer across the ray than along it": b_full["elongation"] >= 1.0,
}
for k, v in gates.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
if not sens["gate_pass"]:
    print("\nStep 9b failed, and it is the gate the transfer claim rests on: the "
          "gradients above\nare verified as arithmetic but not as physics, so nothing "
          "here establishes that the\nsurrogate's geometry sensitivity is the medium's.")
print("\nStill outstanding, and not gated here: 9b and 9c were run at one geometry, "
      "one\nillumination and one Poisson ratio, so they certify this checkpoint at this "
      "operating\npoint rather than the surrogate as such; the out-of-family version of "
      "the same\nmeasurement is notebook 06.")
print("\nNotebook 06: the RingCNN baseline, the out-of-family transfer, and the")
print("lack-of-fit indicator -- the three results the thesis claim rests on.")